# Pipeline Debug Notebook
Full pipeline: Feature Engineering → Preprocessing → Training

**Order:** Run cells top-to-bottom. Each section verifies the previous step before proceeding.

In [ ]:
# ── 0. SETUP ─────────────────────────────────────────────────────────────────
import sys, time
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 50)

CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MODELS    = PROJECT_ROOT / 'models'
TARGET    = config['project']['target_col']          # 'is_churn'
CUTOFF    = config['feature_engineering']['cutoff_date']
RS        = int(config['project']['random_state'])

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'cutoff_date   : {CUTOFF}   <- must be 2017-01-31')
print(f'random_state  : {RS}')
print(f'candidate_models: {config["modeling"]["candidate_models"]}')
print()

# Check package versions
import sklearn, xgboost, lightgbm
print(f'sklearn  {sklearn.__version__}')
print(f'xgboost  {xgboost.__version__}')
print(f'lightgbm {lightgbm.__version__}')

---
## SECTION 1 — Feature Engineering
Runs `run_feature_engineering()` and verifies the output feature frame.

In [ ]:
# ── 1a. Run feature engineering ───────────────────────────────────────────────
from src.features.pipeline import run_feature_engineering

print('Running feature engineering...')
t0 = time.perf_counter()
ff = run_feature_engineering(CONFIG_PATH)
print(f'Done in {time.perf_counter()-t0:.1f}s')

In [ ]:
# ── 1b. Basic checks ─────────────────────────────────────────────────────────
print(f'Shape         : {ff.shape}')
print(f'Columns       : {len(ff.columns)}')
print()

ref_date = ff['analysis_reference_date'].iloc[0] if 'analysis_reference_date' in ff.columns else 'MISSING'
status   = 'OK' if str(ref_date)[:10] == CUTOFF else 'WRONG — leakage risk!'
print(f'analysis_reference_date : {ref_date}  [{status}]')
print()

n_total   = len(ff)
n_churn   = int(ff[TARGET].sum())
n_nochurn = n_total - n_churn
print(f'Total users   : {n_total:,}')
print(f'Churners      : {n_churn:,}  ({100*n_churn/n_total:.2f}%)')
print(f'Non-churners  : {n_nochurn:,}  ({100*n_nochurn/n_total:.2f}%)')
print(f'Imbalance SPW : {n_nochurn/n_churn:.2f}')

In [ ]:
# ── 1c. Date range verification ───────────────────────────────────────────────
print('DATE RANGES (all must be <= 2017-01-31)\n')
for col in ['last_transaction_date', 'last_log_date', 'last_expire_date']:
    if col in ff.columns:
        mn = ff[col].min()
        mx = ff[col].max()
        warn = '  [FUTURE DATA!]' if str(mx)[:10] > CUTOFF else '  [OK]'
        print(f'  {col:30s}: {mn} -> {mx}{warn}')

print()
print('KEY DERIVED FEATURES — Churner vs Non-churner means:')
feat_check = [
    'days_since_last_transaction', 'auto_renew_rate', 'cancel_rate',
    'retention_rate_from_transactions', 'recent_transaction_flag',
    'days_since_last_log', 'trans_count', 'mean_plan_price'
]
rows = []
for f in feat_check:
    if f not in ff.columns:
        continue
    m_c  = ff.loc[ff[TARGET]==1, f].mean()
    m_nc = ff.loc[ff[TARGET]==0, f].mean()
    rows.append({'feature': f, 'churner_mean': m_c, 'non_churner_mean': m_nc,
                 'direction_ok': 'OK' if abs(m_c - m_nc) > 0.001 else 'WARNING: near-zero gap'})
display(pd.DataFrame(rows).set_index('feature'))

In [ ]:
# ── 1d. Dtype audit ───────────────────────────────────────────────────────────
print('DTYPE AUDIT — columns with extension dtypes:\n')
ext_cols = []
for col in ff.columns:
    dt = ff[col].dtype
    if isinstance(dt, (pd.StringDtype, pd.BooleanDtype, pd.CategoricalDtype)):
        ext_cols.append({'column': col, 'dtype': str(dt), 'nulls': ff[col].isna().sum()})

if ext_cols:
    display(pd.DataFrame(ext_cols))
else:
    print('  (none — all columns are standard numpy dtypes)')

print()
print(f'Columns with ANY null  : {ff.isnull().any(axis=1).sum():,} rows')
print()
null_counts = ff.isnull().sum()
print('Top 10 columns by null count:')
display(null_counts[null_counts > 0].sort_values(ascending=False).head(10))

In [ ]:
# ── 1e. Pearson correlation with is_churn ─────────────────────────────────────
print('TOP 20 CORRELATIONS WITH is_churn (absolute value):\n')
num_cols = ff.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]
corrs = ff[num_cols].corrwith(ff[TARGET]).abs().sort_values(ascending=False)

for col, val in corrs.head(20).items():
    bar = '=' * int(val * 50)
    signed = ff[[col, TARGET]].corr().iloc[0, 1]
    sign = '+' if signed >= 0 else '-'
    print(f'  {col:40s}  {sign}{val:.4f}  {bar}')

print()
top_corr = corrs.iloc[0]
if top_corr > 0.85:
    print(f'[WARNING] max correlation = {top_corr:.4f} — potential leakage!')
elif top_corr > 0.6:
    print(f'[CHECK]   High correlation = {top_corr:.4f} — check carefully')
else:
    print(f'[OK]      Max correlation = {top_corr:.4f} — looks reasonable')

---
## SECTION 2 — Preprocessing

In [ ]:
# ── 2a. Split dataset ─────────────────────────────────────────────────────────
from src.features.preprocess import (
    split_dataset, identify_column_groups,
    build_preprocessor, fit_preprocessor, apply_preprocessor,
    build_linear_preprocessor, _sanitise_for_sklearn,
    save_splits, save_preprocessor, DataSplits,
)

print('Splitting dataset...')
train_df, val_df, test_df = split_dataset(ff, config)

print(f'\nSplit sizes:')
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    cr = df[TARGET].mean()
    print(f'  {name:5s}: {len(df):>8,} rows  churn={cr:.4f} ({100*cr:.2f}%)')

X_train_raw = train_df.drop(columns=[TARGET])
X_val_raw   = val_df.drop(columns=[TARGET])
X_test_raw  = test_df.drop(columns=[TARGET])
y_train     = train_df[TARGET].astype('int8')
y_val       = val_df[TARGET].astype('int8')
y_test      = test_df[TARGET].astype('int8')

print(f'\nX_train_raw : {X_train_raw.shape}')
print(f'y_train     : {y_train.shape}  churn_rate={y_train.mean():.4f}')

In [ ]:
# ── 2b. Column groups ─────────────────────────────────────────────────────────
extra_drop = [c for c in ['bd', 'gender', 'city', 'registered_via']
              if c in X_train_raw.columns]
groups = identify_column_groups(X_train_raw, extra_drop=extra_drop)

print(f'Numeric cols    : {len(groups.numeric)}')
print(f'Categorical cols: {len(groups.categorical)}')
print(f'Dropped cols    : {len(groups.drop)}')
print()
print('Numeric (first 10):', groups.numeric[:10])
if len(groups.numeric) > 10:
    print(f'... and {len(groups.numeric)-10} more')
print()
if groups.categorical:
    print('Categorical:', groups.categorical)
else:
    print('Categorical: (none)')
print()
print('Dropped:', sorted(groups.drop))

In [ ]:
# ── 2c. Tree preprocessor (OrdinalEncoder) ────────────────────────────────────
print('Building + fitting tree preprocessor (OrdinalEncoder + RobustScaler)...')
preprocessor = build_preprocessor(groups)
fit_preprocessor(preprocessor, X_train_raw)

X_train = apply_preprocessor(preprocessor, X_train_raw, 'train')
X_val   = apply_preprocessor(preprocessor, X_val_raw,   'val')
X_test  = apply_preprocessor(preprocessor, X_test_raw,  'test')

print(f'\nTransformed shapes:')
print(f'  X_train : {X_train.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')
print()

nan_tr = X_train.isnull().sum().sum()
nan_vl = X_val.isnull().sum().sum()
print(f'NaN in X_train after preprocessing: {nan_tr}  [{"OK" if nan_tr==0 else "ERROR"}]')
print(f'NaN in X_val   after preprocessing: {nan_vl}  [{"OK" if nan_vl==0 else "ERROR"}]')

print()
print('Sample of transformed X_train (first 3 rows, first 5 cols):')
display(X_train.iloc[:3, :5])

In [ ]:
# ── 2d. Leakage / index overlap check ────────────────────────────────────────
train_idx = set(X_train.index)
val_idx   = set(X_val.index)
test_idx  = set(X_test.index)

overlap_tv = train_idx & val_idx
overlap_tt = train_idx & test_idx
overlap_vt = val_idx  & test_idx

print('INDEX OVERLAP CHECK (must all be 0):')
print(f'  train & val  : {len(overlap_tv)}  [{"OK" if not overlap_tv else "OVERLAP ERROR"}]')
print(f'  train & test : {len(overlap_tt)}  [{"OK" if not overlap_tt else "OVERLAP ERROR"}]')
print(f'  val   & test : {len(overlap_vt)}  [{"OK" if not overlap_vt else "OVERLAP ERROR"}]')
total_union = len(train_idx|val_idx|test_idx)
print(f'  union = {total_union:,}  total ff = {len(ff):,}  [{"OK" if total_union==len(ff) else "MISSING ROWS"}]')

print()
print('y alignment check:')
yr_ok = (y_train.index == X_train.index).all()
yv_ok = (y_val.index   == X_val.index).all()
print(f'  y_train index == X_train index: {yr_ok}  [{"OK" if yr_ok else "MISMATCH"}]')
print(f'  y_val   index == X_val   index: {yv_ok}  [{"OK" if yv_ok else "MISMATCH"}]')

In [ ]:
# ── 2e. Linear preprocessor for LR (OHE + RobustScaler) ──────────────────────
print('Building linear preprocessor (OneHotEncoder + RobustScaler) for LR...')
print()

_LINEAR_DROP = {
    TARGET, 'msno', 'analysis_reference_date',
    'registration_init_time', 'last_transaction_date',
    'last_expire_date', 'last_log_date',
    'bd', 'gender', 'city', 'registered_via',
}

feat_cols_lr = [c for c in ff.columns if c not in _LINEAR_DROP]
groups_lr    = identify_column_groups(train_df[feat_cols_lr], extra_drop=list(_LINEAR_DROP))

print(f'Categorical columns found: {len(groups_lr.categorical)}')
if groups_lr.categorical:
    cat_info = []
    for col in groups_lr.categorical:
        n_unique = train_df[col].nunique(dropna=True)
        cat_info.append({
            'column': col,
            'dtype': str(train_df[col].dtype),
            'n_unique': n_unique,
            'OHE_safe': 'YES' if n_unique <= 50 else 'NO (will be dropped)',
        })
    display(pd.DataFrame(cat_info).set_index('column'))
else:
    print('  (none)')
print()

MAX_OHE = 50
lr_pre = build_linear_preprocessor(
    groups_lr,
    X_ref=train_df[feat_cols_lr],
    max_ohe_categories=MAX_OHE,
)

train_lr_clean = _sanitise_for_sklearn(train_df[feat_cols_lr])
val_lr_clean   = _sanitise_for_sklearn(val_df[feat_cols_lr])

t0 = time.perf_counter()
X_train_lr = lr_pre.fit_transform(train_lr_clean)
X_val_lr   = lr_pre.transform(val_lr_clean)

y_train_lr = train_df[TARGET].values.astype('int8')
y_val_lr   = val_df[TARGET].values.astype('int8')

print(f'Done in {time.perf_counter()-t0:.1f}s')
print(f'\nX_train_lr shape : {X_train_lr.shape}')
print(f'X_val_lr   shape : {X_val_lr.shape}')
print(f'y_train_lr shape : {y_train_lr.shape}  churn_rate={y_train_lr.mean():.4f}')
print(f'y_val_lr   shape : {y_val_lr.shape}    churn_rate={y_val_lr.mean():.4f}')
print()

y_train_arr = y_train.values.astype('int8')
y_val_arr   = y_val.values.astype('int8')
match_tr = (y_train_lr == y_train_arr).all()
match_vl = (y_val_lr   == y_val_arr).all()
print(f'y_train_lr == y_train : {match_tr}  [{"OK" if match_tr else "MISMATCH — LR labels wrong!"}]')
print(f'y_val_lr   == y_val   : {match_vl}  [{"OK" if match_vl else "MISMATCH — LR labels wrong!"}]')

In [ ]:
# ── 2f. Save splits + preprocessor ───────────────────────────────────────────
splits = DataSplits(
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_train=y_train, y_val=y_val, y_test=y_test,
)
splits.log_shapes()

save_splits(splits, PROCESSED)
save_preprocessor(preprocessor, MODELS / 'preprocessor.pkl')

print('\n[OK] Preprocessing complete. Splits saved to data/processed/')

---
## SECTION 3 — Training

In [ ]:
# ── 3a. Imports and metrics helper ────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score,
)
import xgboost as xgb
import lightgbm as lgb

def compute_metrics(y_true, y_score, threshold=0.5, label=''):
    y_pred = (y_score >= threshold).astype(int)
    n_top  = max(1, int(len(y_true) * 0.10))
    top_ix = np.argsort(y_score)[::-1][:n_top]
    lift   = np.asarray(y_true)[top_ix].mean() / np.asarray(y_true).mean()
    return {
        'split':     label,
        'auc_roc':   round(roc_auc_score(y_true, y_score), 4),
        'auc_pr':    round(average_precision_score(y_true, y_score), 4),
        'f1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
        'precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'lift@10%':  round(lift, 4),
    }

spw = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f'Class distribution | neg={y_train.value_counts()[0]:,}  pos={y_train.value_counts()[1]:,}  spw={spw:.2f}')
all_results = []

In [ ]:
# ── 3b. [1/4] Logistic Regression ─────────────────────────────────────────────
# Reload config so changes to config.yaml are picked up without kernel restart.
with open(CONFIG_PATH) as _f:
    config = yaml.safe_load(_f)

lr_cfg = config['logistic_regression']

print('=' * 60)
print('[1/4] Logistic Regression')
print('=' * 60)
print(f"  solver={lr_cfg['solver']}  penalty={lr_cfg['penalty']}  "
      f"C={lr_cfg['C']}  max_iter={lr_cfg['max_iter']}")
print(f'  X_train_lr : {X_train_lr.shape}')
print(f'  y_train_lr : {y_train_lr.shape}  churn={y_train_lr.mean():.4f}')

t0 = time.perf_counter()
lr = LogisticRegression(
    solver=lr_cfg['solver'],
    penalty=lr_cfg['penalty'],
    C=float(lr_cfg['C']),
    max_iter=int(lr_cfg['max_iter']),
    class_weight=lr_cfg['class_weight'],
    random_state=RS,
)
lr.fit(X_train_lr, y_train_lr)
elapsed = time.perf_counter() - t0

lr_tr_score = lr.predict_proba(X_train_lr)[:, 1]
lr_vl_score = lr.predict_proba(X_val_lr)[:, 1]

m_tr = compute_metrics(y_train_lr, lr_tr_score, label='train')
m_vl = compute_metrics(y_val_lr,   lr_vl_score, label='val')
display(pd.DataFrame([m_tr, m_vl]))
print(f'  Done in {elapsed:.1f}s  |  n_iter={lr.n_iter_[0]}')

# Diagnostics
print()
print('DIAGNOSTIC — predict_proba distribution:')
print(f'  std  : {lr_vl_score.std():.4f}  (want >> 0.05)')
print(f'  range: [{lr_vl_score.min():.4f}, {lr_vl_score.max():.4f}]')
print()
print('DIAGNOSTIC — mean P(churn) by actual label:')
print(f'  actual churn=1 : {lr_vl_score[y_val_lr==1].mean():.4f}  (want HIGH > 0.5)')
print(f'  actual churn=0 : {lr_vl_score[y_val_lr==0].mean():.4f}  (want LOW < 0.5)')

# Coefficients
coef = lr.coef_[0]
try:
    feat_names_lr = lr_pre.get_feature_names_out()
except Exception:
    feat_names_lr = [f'f{i}' for i in range(len(coef))]

coef_s   = pd.Series(coef, index=feat_names_lr).sort_values()
n_zero   = (coef_s == 0).sum()
n_nonzero = len(coef_s) - n_zero
print()
print(f'Coefficients: {n_nonzero} non-zero / {len(coef_s)} total  (L1 zeros: {n_zero})')
print('Top 5 NEGATIVE (predict non-churn):')
display(coef_s.head(5).to_frame('coef'))
print('Top 5 POSITIVE (predict churn):')
display(coef_s.tail(5).to_frame('coef'))

all_results.append({'model': 'logistic_regression',
                    'train_auc_pr': m_tr['auc_pr'], 'val_auc_pr': m_vl['auc_pr'],
                    'train_auc_roc': m_tr['auc_roc'], 'val_auc_roc': m_vl['auc_roc'],
                    'val_f1': m_vl['f1'], 'val_precision': m_vl['precision'],
                    'val_recall': m_vl['recall'], 'val_lift': m_vl['lift@10%'],
                    'best_iter': lr.n_iter_[0]})

In [ ]:
# ── 3c. [2/4] Random Forest ───────────────────────────────────────────────────
print('=' * 60)
print('[2/4] Random Forest')
print('=' * 60)
print(f'  X_train : {X_train.shape}')

t0 = time.perf_counter()
rf = RandomForestClassifier(
    n_estimators=int(config['random_forest']['n_estimators']),
    max_depth=config['random_forest']['max_depth'],
    min_samples_split=int(config['random_forest']['min_samples_split']),
    min_samples_leaf=int(config['random_forest']['min_samples_leaf']),
    class_weight=config['random_forest']['class_weight'],
    n_jobs=-1,
    random_state=RS,
)
rf.fit(X_train, y_train)
elapsed = time.perf_counter() - t0

rf_tr_score = rf.predict_proba(X_train)[:, 1]
rf_vl_score = rf.predict_proba(X_val)[:, 1]

m_tr = compute_metrics(y_train, rf_tr_score, label='train')
m_vl = compute_metrics(y_val,   rf_vl_score, label='val')
display(pd.DataFrame([m_tr, m_vl]))
print(f'  Done in {elapsed:.1f}s')

print()
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('Top 15 feature importances (RF):')
for feat, val in imp.head(15).items():
    bar = '=' * int(val / imp.max() * 40)
    print(f'  {feat:40s}  {val:.4f}  {bar}')

all_results.append({'model': 'random_forest',
                    'train_auc_pr': m_tr['auc_pr'], 'val_auc_pr': m_vl['auc_pr'],
                    'train_auc_roc': m_tr['auc_roc'], 'val_auc_roc': m_vl['auc_roc'],
                    'val_f1': m_vl['f1'], 'val_precision': m_vl['precision'],
                    'val_recall': m_vl['recall'], 'val_lift': m_vl['lift@10%'],
                    'best_iter': 0})

In [ ]:
# ── 3d. [3/4] XGBoost ─────────────────────────────────────────────────────────
print('=' * 60)
print('[3/4] XGBoost')
print('=' * 60)

t0 = time.perf_counter()
xgb_model = xgb.XGBClassifier(
    n_estimators=int(config['xgboost']['n_estimators']),
    learning_rate=float(config['xgboost']['learning_rate']),
    max_depth=int(config['xgboost']['max_depth']),
    subsample=float(config['xgboost']['subsample']),
    colsample_bytree=float(config['xgboost']['colsample_bytree']),
    tree_method=config['xgboost']['tree_method'],
    eval_metric=config['xgboost']['eval_metric'],
    early_stopping_rounds=int(config['xgboost']['early_stopping_rounds']),
    scale_pos_weight=spw,
    random_state=RS,
    n_jobs=-1,
    verbosity=0,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=int(config['xgboost']['log_eval_period']),
)
elapsed = time.perf_counter() - t0
best_iter = int(xgb_model.best_iteration)

xgb_tr_score = xgb_model.predict_proba(X_train)[:, 1]
xgb_vl_score = xgb_model.predict_proba(X_val)[:, 1]

m_tr = compute_metrics(y_train, xgb_tr_score, label='train')
m_vl = compute_metrics(y_val,   xgb_vl_score, label='val')
display(pd.DataFrame([m_tr, m_vl]))
print(f'  Done in {elapsed:.1f}s  |  best_iter={best_iter}')

print()
imp_xgb = pd.Series(
    xgb_model.get_booster().get_score(importance_type='gain'),
).sort_values(ascending=False)
print('Top 15 feature importances (XGB, gain):')
for feat, val in imp_xgb.head(15).items():
    bar = '=' * int(val / imp_xgb.max() * 40)
    print(f'  {feat:40s}  {val:.1f}  {bar}')

all_results.append({'model': 'xgboost',
                    'train_auc_pr': m_tr['auc_pr'], 'val_auc_pr': m_vl['auc_pr'],
                    'train_auc_roc': m_tr['auc_roc'], 'val_auc_roc': m_vl['auc_roc'],
                    'val_f1': m_vl['f1'], 'val_precision': m_vl['precision'],
                    'val_recall': m_vl['recall'], 'val_lift': m_vl['lift@10%'],
                    'best_iter': best_iter})

In [ ]:
# ── 3e. [4/4] LightGBM ────────────────────────────────────────────────────────
print('=' * 60)
print('[4/4] LightGBM')
print('=' * 60)

lgb_metric = config['lightgbm']['metric']

t0 = time.perf_counter()
lgb_model = lgb.LGBMClassifier(
    n_estimators=int(config['lightgbm']['n_estimators']),
    learning_rate=float(config['lightgbm']['learning_rate']),
    num_leaves=int(config['lightgbm']['num_leaves']),
    max_depth=int(config['lightgbm']['max_depth']),
    min_child_samples=int(config['lightgbm']['min_child_samples']),
    reg_alpha=float(config['lightgbm']['reg_alpha']),
    reg_lambda=float(config['lightgbm']['reg_lambda']),
    subsample=float(config['lightgbm']['subsample']),
    colsample_bytree=float(config['lightgbm']['colsample_bytree']),
    scale_pos_weight=spw,
    metric=lgb_metric,
    random_state=RS,
    n_jobs=-1,
    verbose=-1,
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=int(config['lightgbm']['early_stopping_rounds']), verbose=False),
        lgb.log_evaluation(period=int(config['lightgbm']['log_eval_period'])),
    ],
)
elapsed = time.perf_counter() - t0
best_iter = int(lgb_model.best_iteration_) if lgb_model.best_iteration_ else int(config['lightgbm']['n_estimators'])

lgb_tr_score = lgb_model.predict_proba(X_train)[:, 1]
lgb_vl_score = lgb_model.predict_proba(X_val)[:, 1]

m_tr = compute_metrics(y_train, lgb_tr_score, label='train')
m_vl = compute_metrics(y_val,   lgb_vl_score, label='val')
display(pd.DataFrame([m_tr, m_vl]))
print(f'  Done in {elapsed:.1f}s  |  best_iter={best_iter}')

print()
imp_lgb = pd.Series(
    lgb_model.feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)
print('Top 15 feature importances (LGB, split):')
for feat, val in imp_lgb.head(15).items():
    bar = '=' * int(val / max(imp_lgb.max(), 1) * 40)
    print(f'  {feat:40s}  {val}  {bar}')

all_results.append({'model': 'lightgbm',
                    'train_auc_pr': m_tr['auc_pr'], 'val_auc_pr': m_vl['auc_pr'],
                    'train_auc_roc': m_tr['auc_roc'], 'val_auc_roc': m_vl['auc_roc'],
                    'val_f1': m_vl['f1'], 'val_precision': m_vl['precision'],
                    'val_recall': m_vl['recall'], 'val_lift': m_vl['lift@10%'],
                    'best_iter': best_iter})

---
## SECTION 4 — Final Comparison & Diagnostics

In [ ]:
# ── 4a. Model comparison table ────────────────────────────────────────────────
comp = pd.DataFrame(all_results).sort_values('val_auc_pr', ascending=False).reset_index(drop=True)

print('=' * 80)
print('  MODEL COMPARISON — Validation Set  (ranked by AUC-PR)')
print('=' * 80)
display(comp)

print()
print('OVERFITTING ANALYSIS (train AUC-PR - val AUC-PR):')
print('-' * 60)
for _, row in comp.iterrows():
    gap  = row['train_auc_pr'] - row['val_auc_pr']
    flag = '[OK]' if gap < 0.08 else ('[MILD]' if gap < 0.15 else '[OVERFIT]')
    print(f"  {row['model']:25s}  train={row['train_auc_pr']:.4f}  val={row['val_auc_pr']:.4f}  gap={gap:+.4f}  {flag}")

champion = comp.iloc[0]
print()
print(f"Champion : {champion['model']}")
print(f"  AUC-PR : {champion['val_auc_pr']:.4f}")
print(f"  AUC-ROC: {champion['val_auc_roc']:.4f}")

In [ ]:
# ── 4b. Champion on TEST set ──────────────────────────────────────────────────
print(f"Evaluating champion ({champion['model']}) on TEST set (held-out)...")
print()

champ_name = champion['model']
champ_mdl  = {'logistic_regression': lr,
               'random_forest': rf,
               'xgboost': xgb_model,
               'lightgbm': lgb_model}[champ_name]

if champ_name == 'logistic_regression':
    X_test_lr_clean = _sanitise_for_sklearn(test_df[feat_cols_lr])
    X_test_input    = lr_pre.transform(X_test_lr_clean)
    y_test_input    = test_df[TARGET].values.astype('int8')
else:
    X_test_input  = X_test
    y_test_input  = y_test

test_score = champ_mdl.predict_proba(X_test_input)[:, 1]
m_test     = compute_metrics(y_test_input, test_score, label='test')
display(pd.DataFrame([m_test]))

val_pr  = champion['val_auc_pr']
test_pr = m_test['auc_pr']
diff    = abs(val_pr - test_pr)
print()
print(f'val AUC-PR  : {val_pr:.4f}')
print(f'test AUC-PR : {test_pr:.4f}')
print(f'difference  : {diff:.4f}  [{"OK - stable" if diff < 0.03 else "WARN - val/test gap is large"}]')